In [4]:
# ImportError: Unable to find a usable engine; tried using: 'pyarrow', 'fastparquet'.
# A suitable version of pyarrow or fastparquet is required for parquet support.
# Trying to import the above resulted in these errors:
#  - Missing optional dependency 'pyarrow'. pyarrow is required for parquet support. Use pip or conda to install pyarrow.
#  - Missing optional dependency 'fastparquet'. fastparquet is required for parquet support. Use pip or conda to install fastparquet.
# Output is truncated. View as a scrollable element or open in a text editor. Adjust cell output settings... 
!pip install pyarrow
!pip install fastparquet

   ---------------------------------------- 0.0/27.5 MB ? eta -:--:--
   --- ------------------------------------ 2.6/27.5 MB 16.7 MB/s eta 0:00:02
   --------- ------------------------------ 6.3/27.5 MB 16.8 MB/s eta 0:00:02
   ------------- -------------------------- 9.2/27.5 MB 15.4 MB/s eta 0:00:02
   ----------------- ---------------------- 12.1/27.5 MB 15.1 MB/s eta 0:00:02
   --------------------- ------------------ 14.9/27.5 MB 14.7 MB/s eta 0:00:01
   ------------------------- -------------- 17.8/27.5 MB 14.6 MB/s eta 0:00:01
   ----------------------------- ---------- 20.4/27.5 MB 14.2 MB/s eta 0:00:01
   --------------------------------- ------ 23.3/27.5 MB 14.1 MB/s eta 0:00:01
   ------------------------------------- -- 26.0/27.5 MB 13.8 MB/s eta 0:00:01
   ---------------------------------------  27.3/27.5 MB 13.8 MB/s eta 0:00:01
   ---------------------------------------- 27.5/27.5 MB 13.2 MB/s  0:00:02



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


   ---------------------------------------- 0.0/669.8 kB ? eta -:--:--
   ---------------------------------------- 669.8/669.8 kB 13.1 MB/s  0:00:00
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ---------------------------------------- 1.7/1.7 MB 13.2 MB/s  0:00:00

   -------------------- ------------------- 1/2 [fastparquet]
   -------------------- ------------------- 1/2 [fastparquet]
   ---------------------------------------- 2/2 [fastparquet]




[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import io
import requests
import pandas as pd

def load_data(*args, **kwargs):
    year = kwargs.get('year', 2025)
    month = kwargs.get('month', '01')

    url = f'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_{year}-{month}.parquet'

    # Some institutional networks block generic urllib traffic and return HTTP 403.
    headers = {
        'User-Agent': kwargs.get(
            'user_agent',
            'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 '
            '(KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36'
        )
    }

    response = requests.get(url, headers=headers, timeout=60)
    response.raise_for_status()

    parquet_buffer = io.BytesIO(response.content)

    engine = kwargs.get('engine', 'fastparquet')
    try:
        datos_crudos = pd.read_parquet(parquet_buffer, engine=engine)
    except Exception:
        parquet_buffer.seek(0)
        datos_crudos = pd.read_parquet(parquet_buffer, engine='fastparquet')

    return datos_crudos

data = load_data(year=2015, month='01')
data.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee
0,1,2015-01-01 00:11:33,2015-01-01 00:16:48,1,1.0,1,N,41,166,1,5.7,0.5,0.5,1.40,0.0,0.0,8.40,NaN,NaN
1,1,2015-01-01 00:18:24,2015-01-01 00:24:20,1,0.9,1,N,166,238,3,6.0,0.5,0.5,0.00,0.0,0.0,7.30,NaN,NaN
2,1,2015-01-01 00:26:19,2015-01-01 00:41:06,1,3.5,1,N,238,162,1,13.2,0.5,0.5,2.90,0.0,0.0,17.40,NaN,NaN
3,1,2015-01-01 00:45:26,2015-01-01 00:53:20,1,2.1,1,N,162,263,1,8.2,0.5,0.5,2.37,0.0,0.0,11.87,NaN,NaN
4,1,2015-01-01 00:59:21,2015-01-01 01:05:24,1,1.0,1,N,236,141,3,6.0,0.5,0.5,0.00,0.0,0.0,7.30,NaN,NaN


In [4]:
data.columns

Index(['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime',
       'passenger_count', 'trip_distance', 'RatecodeID', 'store_and_fwd_flag',
       'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra',
       'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge',
       'total_amount', 'congestion_surcharge', 'airport_fee'],
      dtype='object')

In [5]:
def transform(data, *args, **kwargs):
    data.columns = [col.lower() for col in data.columns]
    
    data.rename(columns={
        'vendorid': 'vendor_id',
        'ratecodeid': 'rate_code_id',
        'pulocationid': 'pu_location_id',
        'dolocationid': 'do_location_id'
    }, inplace = True)

    return data

data = transform(data)
data.columns

Index(['vendor_id', 'tpep_pickup_datetime', 'tpep_dropoff_datetime',
       'passenger_count', 'trip_distance', 'rate_code_id',
       'store_and_fwd_flag', 'pu_location_id', 'do_location_id',
       'payment_type', 'fare_amount', 'extra', 'mta_tax', 'tip_amount',
       'tolls_amount', 'improvement_surcharge', 'total_amount',
       'congestion_surcharge', 'airport_fee'],
      dtype='object')

In [6]:
data.head()

,vendor_id,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,rate_code_id,store_and_fwd_flag,pu_location_id,do_location_id,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee
0,1,2015-01-01 00:11:33,2015-01-01 00:16:48,1,1.0,1,N,41,166,1,5.7,0.5,0.5,1.40,0.0,0.0,8.40,NaN,NaN
1,1,2015-01-01 00:18:24,2015-01-01 00:24:20,1,0.9,1,N,166,238,3,6.0,0.5,0.5,0.00,0.0,0.0,7.30,NaN,NaN
2,1,2015-01-01 00:26:19,2015-01-01 00:41:06,1,3.5,1,N,238,162,1,13.2,0.5,0.5,2.90,0.0,0.0,17.40,NaN,NaN
3,1,2015-01-01 00:45:26,2015-01-01 00:53:20,1,2.1,1,N,162,263,1,8.2,0.5,0.5,2.37,0.0,0.0,11.87,NaN,NaN
4,1,2015-01-01 00:59:21,2015-01-01 01:05:24,1,1.0,1,N,236,141,3,6.0,0.5,0.5,0.00,0.0,0.0,7.30,NaN,NaN


In [20]:
rows = data.shape[0]
chunksize = 500000

print(f'Total rows: {rows}')
print(f'Chunk size: {chunksize}')

print("Replace")

print(f'Chunk {0//chunksize + 1}: {chunksize} rows')

print("Append")

for i in range(chunksize, rows, chunksize):
    chunk = data.iloc[i:i+chunksize]
    print(f'Chunk {i//chunksize + 1}: {chunk.shape[0]} rows')

Total rows: 12741035
Chunk size: 500000
Replace
Chunk 1: 500000 rows
Append
Chunk 2: 500000 rows
Chunk 3: 500000 rows
Chunk 4: 500000 rows
Chunk 5: 500000 rows
Chunk 6: 500000 rows
Chunk 7: 500000 rows
Chunk 8: 500000 rows
Chunk 9: 500000 rows
Chunk 10: 500000 rows
Chunk 11: 500000 rows
Chunk 12: 500000 rows
Chunk 13: 500000 rows
Chunk 14: 500000 rows
Chunk 15: 500000 rows
Chunk 16: 500000 rows
Chunk 17: 500000 rows
Chunk 18: 500000 rows
Chunk 19: 500000 rows
Chunk 20: 500000 rows
Chunk 21: 500000 rows
Chunk 22: 500000 rows
Chunk 23: 500000 rows
Chunk 24: 500000 rows
Chunk 25: 500000 rows
Chunk 26: 241035 rows
